# 03 · 軽いモデルと共通ヘッド

主ラベルは24時間以降〜90日後の自発再聴（L=1、両端含む）。L=7・14は必須感度分析、L=3も残します。学習用モデルは29日目まで、検証・テスト用は149日目までで固定します。

MostPop・DecayPopは公式の再生率50%以上の前処理とDecayPopの7候補を使用。ACT-Rは本文で規定するB/S/Pを、各成分のsoftmax後に合算し、T2はS/Pだけにします。参照時刻は起点+L日、入力は起点まで。音響の好み表現は過去の自発再生の平均です。

In [1]:
import numpy as np
import polars as pl
from IPython.display import display
from relisten import config
from relisten.light_models import build_light_features
from relisten.evaluation import evaluate_features, LIGHT_MODELS
np.random.seed(config.SEED)

## 入力スコア

音響ベクトルは正規化列のみ取得して50M曲に絞ります。ACT-Rの拡散活性は30分セッションの共起lift、部分一致は直前曲との音響類似度です。正規化する候補集合は起点以前に聴いた曲＋対象曲だけで、未来の候補集合は使いません。

In [2]:
features = build_light_features()
display(features.select("split", "model_end_day").group_by("split", "model_end_day").len())

day 29: audio/ACT-R 0/4660 users


day 29: audio/ACT-R 500/4660 users


day 29: audio/ACT-R 1000/4660 users


day 29: audio/ACT-R 1500/4660 users


day 29: audio/ACT-R 2000/4660 users


...

day 149: audio/ACT-R 5000/6526 users


day 149: audio/ACT-R 5500/6526 users


day 149: audio/ACT-R 6000/6526 users


day 149: audio/ACT-R 6500/6526 users


split,model_end_day,len
str,i32,u32
"""train""",29,1285567
"""test""",149,2165342
"""validation""",149,848995


## 同一条件で評価

スコア単独とユーザー基礎率＋活動日数に加えた場合を比較します。標準化とロジスティック回帰の学習は学習ペアだけで行います。C=1を事前固定。PR-AUCはaverage precision。主指標は正負の両方を持つ各ユーザーのAUCの等重み平均、信頼区間はユーザー単位の1000回ブートストラップです。Bは同じ起点・ヘッドでラベルだけを替えます。

In [3]:
metrics = evaluate_features(features, LIGHT_MODELS)
display(metrics.filter((pl.col("lower_days")==config.LOWER_BOUND_DAYS) & (pl.col("route")=="C") & (pl.col("split")=="test")).sort("task", "protocol", "user_auc", descending=[False,False,True]))

evaluated L=1 T2 C


evaluated L=1 T2 B


evaluated L=1 T1 C


evaluated L=1 T1 B


evaluated L=3 T2 C


...

evaluated L=14 T2 C


evaluated L=14 T2 B


evaluated L=14 T1 C


evaluated L=14 T1 B


model,protocol,task,route,split,lower_days,is_main_lower,pairs,users,auc_users,positive_rate,user_auc,user_auc_ci_low,user_auc_ci_high,global_auc,pr_auc,log_loss,brier
str,str,str,str,str,i64,bool,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""UserBaseline""","""baseline""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.5,0.5,0.5,0.614387,0.263604,0.484497,0.154408
"""ACT-R""","""baseline_plus_score""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.616118,0.611034,0.621153,0.620661,0.270846,0.482882,0.153781
"""AudioCosine""","""baseline_plus_score""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.550597,0.545987,0.555191,0.630025,0.271115,0.476122,0.152077
"""DecayPop""","""baseline_plus_score""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.536516,0.531607,0.541383,0.621529,0.269933,0.482595,0.153669
"""MostPop""","""baseline_plus_score""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.529671,0.524748,0.534256,0.591864,0.250853,0.580512,0.185497
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""MostPop""","""baseline_plus_score""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.507429,0.502166,0.512431,0.586049,0.12806,0.380057,0.107423
"""AudioCosine""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.561862,0.557,0.566788,0.596227,0.129059,0.31339,0.08637
"""DecayPop""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.519963,0.514678,0.524862,0.526237,0.122383,0.3161,0.086818


## 必須の感度分析

主ラベルを結果で変えません。L=7・14を含む全結果をCSVに保存します。

In [4]:
display(metrics.filter((pl.col("route")=="C") & (pl.col("task")=="T2") & (pl.col("split")=="test")))

model,protocol,task,route,split,lower_days,is_main_lower,pairs,users,auc_users,positive_rate,user_auc,user_auc_ci_low,user_auc_ci_high,global_auc,pr_auc,log_loss,brier
str,str,str,str,str,i64,bool,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""UserBaseline""","""baseline""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.5,0.5,0.5,0.607669,0.135043,0.31237,0.086232
"""MostPop""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.507429,0.502166,0.512431,0.519231,0.115138,0.385707,0.108432
"""MostPop""","""baseline_plus_score""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.507429,0.502166,0.512431,0.586049,0.12806,0.380057,0.107423
"""DecayPop""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.519963,0.514678,0.524862,0.526237,0.122383,0.3161,0.086818
"""DecayPop""","""baseline_plus_score""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.51992,0.514608,0.524804,0.621509,0.145277,0.311573,0.086042
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""DecayPop""","""baseline_plus_score""","""T2""","""C""","""test""",14,false,905038,4518,3820,0.084515,0.522445,0.516942,0.527364,0.621738,0.126971,0.284875,0.076604
"""AudioCosine""","""score_only""","""T2""","""C""","""test""",14,false,905038,4518,3820,0.084515,0.564298,0.558955,0.569559,0.595434,0.113425,0.286663,0.076868
"""AudioCosine""","""baseline_plus_score""","""T2""","""C""","""test""",14,false,905038,4518,3820,0.084515,0.564298,0.558955,0.569559,0.627027,0.125768,0.283127,0.076359
